In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import spacy
import re
import contractions
from textblob import TextBlob
from nltk.tokenize import word_tokenize


### 1. Load the document (.txt)

In [8]:
data=open('data.txt').read()

### 2. Text Normalization


#### a. converting all the character into lowercase

In [9]:
data=data.lower()

#### b. removing Extra space

In [10]:
data=re.sub(r'\s{2,}','',data)

- removing numbers like 1.

In [11]:
# data=re.sub(r'')

#### c. Contraction

In [12]:
data=contractions.fix(data)  #check the return type before use

#### d. REmoving the punctuation and spl characters

In [13]:
data=re.sub(r'[^0-9a-zA-Z\s]','',data)

#### e. Textblob

- we use textblob to correct the sentence ie. spelling correction

In [14]:
values=TextBlob(data).correct().raw_sentences
data=' '.join(values)


#### f. Spacy lamentization

In [15]:
nlp=spacy.load('en_core_web_sm')
tokens=nlp(data)
update_tokens=[token.lemma_ for token in tokens if not token.is_stop]
data=' '.join(update_tokens).strip()  # we use strip to extra spaces

#### g. chunking(converting the doc into chunks [doc-->chunk])

In [ ]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40
)
# Generate chunks from the lemmatized text
'''chunks = splitter.split_text(data)'''  # it return all chunks
chunks = splitter.create_documents([data])
chunks

([Document(metadata={}, page_content='machine learning branch artificial intelligence focus enable computer learn pattern datum \n instead explicitly program rule machine learning algorithm learn example use example prediction'),
  Document(metadata={}, page_content='machine learning important technology industry \n healthcare finance education transportation retail entertainment cybersecurity \n basic idea machine learning simple'),
  Document(metadata={}, page_content='basic idea machine learning simple \n provide datum algorithm allow algorithm learn pattern use train model prediction new datum \n machine learning divide major category'),
  Document(metadata={}, page_content='common category supervise learn supervise learning semisupervise learning reinforcement learn \n type learning solve different kind problem'),
  Document(metadata={}, page_content='choice learn method depend type datum available objective project \n supervise learning type machine learning model learn label dat

In [24]:
print(chunks[0].page_content)
# print(type(chunks[0]))

machine learning branch artificial intelligence focus enable computer learn pattern datum 
 instead explicitly program rule machine learning algorithm learn example use example prediction


In [ ]:
chunks[0].metadata={'file_name':'data.txt'}  # we add informations using the meta data
chunks

[Document(metadata={'file_name': 'data.txt'}, page_content='machine learning branch artificial intelligence focus enable computer learn pattern datum \n instead explicitly program rule machine learning algorithm learn example use example prediction'),
 Document(metadata={}, page_content='machine learning important technology industry \n healthcare finance education transportation retail entertainment cybersecurity \n basic idea machine learning simple'),
 Document(metadata={}, page_content='basic idea machine learning simple \n provide datum algorithm allow algorithm learn pattern use train model prediction new datum \n machine learning divide major category'),
 Document(metadata={}, page_content='common category supervise learn supervise learning semisupervise learning reinforcement learn \n type learning solve different kind problem'),
 Document(metadata={}, page_content='choice learn method depend type datum available objective project \n supervise learning type machine learning mod

In [17]:
print(len(chunks))

139


#### h. chunk embeddings (converting the chunks to vectors)

In [ ]:
embedding_model=HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-miniLM-L6-V2'  #to convert the chunks into vector
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [32]:
vectordb=FAISS.from_documents(documents=chunks,embedding=embedding_model)
vectordb

### Retrival 

In [ ]:
user_query='what is machine learning?'
r_chunks=vectordb.similarity_search(user_query)



machine learning important technology industry 
 healthcare finance education transportation retail entertainment cybersecurity 
 basic idea machine learning simple
domain knowledge helps identify meaningful feature 
 communication skill help explain result slaveholder 
 skill contribute successful machine learn projectsmachine learning continue evolve rapidly
porch widely research industry 
 tensorflow widely develop machine learning deep learning application
explainable ai technique attempt provide insight model predictionsthe machine learning lifecycle include stage 
 process begin understand business problem 
 step collect relevant datum


### combining the sentence

In [42]:
updated_r_chunks=set()

for chunk in r_chunks:
    updated_r_chunks.add(chunk.page_content)

r_text='\n'.join(updated_r_chunks)
r_text


'machine learning important technology industry \n healthcare finance education transportation retail entertainment cybersecurity \n basic idea machine learning simple\nexplainable ai technique attempt provide insight model predictionsthe machine learning lifecycle include stage \n process begin understand business problem \n step collect relevant datum\nporch widely research industry \n tensorflow widely develop machine learning deep learning application\ndomain knowledge helps identify meaningful feature \n communication skill help explain result slaveholder \n skill contribute successful machine learn projectsmachine learning continue evolve rapidly'

In [ ]:
# -----------------------------------------------------------------------

In [ ]:
def rag_query(query,k=2):
    query_embedding=embedding_model.encode(query).astype('float32')
    query_embedding=query_embedding.reshape(1,-1)
    faiss.normalize_L2(query_embedding)
    print(query_embedding.shape)
    values=distance,index=index_faiss_db.search(query_embedding,k=k)
    print(values)

    R_chunks=[chunks[i] for i in index[0]] 
    R_str=' '.join(R_chunks)
    prompt= f''' 
                you're an helpful assistant 
                Assigned Task for you: Structure my output => {R_str}
                Note:
                1) Don't add extra contents just structure mentioned output.
                2) If there is mistake in output correct or else keep the original output
    
    '''


(1, 384)
(array([[0.57110596, 0.53118783]], dtype=float32), array([[2, 5]], dtype=int64))
machine learning important technology industry 
 healthcare finance education transportation retail entertainment cybersecurity 
 basic idea machine learning simple explainable ai technique attempt provide insight model predictionsthe machine learning lifecycle include stage 
 process begin understand business problem 
 step collect relevant datum


#### used hugging face

In [ ]:
def r_search(query,k=3):
    query_embeddings = embedding_model.encode(query).astype('float32')
    query_embeddings = query_embeddings.reshape(1,-1)
    faiss.normalize_L2(query_embeddings)
    print(query_embeddings.shape)
    distance,index = index_faiss_db.search(query_embeddings,k=k)
    R_chunks = [chunks[i] for i in index[0]]
    R_str = ' '.join(R_chunks)
    return R_str
def g_text(r_search):
        import os
        import requests
    
        API_URL = "https://router.huggingface.co/v1/chat/completions"
    
        headers = {
            "Authorization": f"Bearer {os.environ['HF_TOKEN']}",
        }
        def query(payload):
            response = requests.post(API_URL, headers=headers, json=payload)
            return response
        prompt = f'''
                    You're an helpful assistant
                    Assigned Task for you : Structure my output => {r_search}
                    Note : 
                    1) Don't add extra contents just structure mentioned output.
                    2) If there is mistake in output correct or else keep the original output
                    with structured result.
            '''
        response = query({
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "model": "deepseek-ai/DeepSeek-R1:novita"
        })
    
        return response
user_prompt = 'Explain Machine Learning ?'
user_prompt = re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)

r_response = r_search(user_prompt)
g_response = g_text(r_response)
print(g_response)

(1, 384)
<Response [200]>
